# Exercise 2: File Format Showdown — VCF vs Parquet
## BINFX410 — Chapter 10: Data Lakes, Warehouses, and Lakehouses

---

## Learning Objectives

- Explain why columnar file formats (Parquet) outperform row-based formats (VCF/CSV) for analytics
- Measure the difference in bytes scanned and query time between VCF and Parquet in Athena
- Convert a VCF file to Parquet using pandas and pyarrow
- Calculate the real dollar cost of a query in both formats
- Articulate when VCF format is still appropriate (hint: it's not going away)

**Estimated time:** 45 minutes  
**Dataset:** ClinVar summary variants — Parquet at `s3://aws-roda-hcls-datalake/clinvar_summary_variants/`  
**AWS services:** S3, Athena, Glue Data Catalog

## Background: The VCF Format Was Designed for Bioinformatics, Not Analytics

The **Variant Call Format (VCF)** was designed in 2010 for interoperability between variant callers and annotation tools — not for cloud analytics. It has some properties that make it expensive to query at scale:

### VCF is row-oriented
```
##fileformat=VCFv4.2
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">
#CHROM  POS     ID     REF  ALT  QUAL  FILTER  INFO
chr1    925952  rs123  G    A    50.2  PASS    DP=42;AF=0.45;CLNSIG=Pathogenic
chr1    931279  rs456  A    G    88.1  PASS    DP=67;AF=0.52;CLNSIG=Benign
```

To answer `SELECT COUNT(*) WHERE CLNSIG='Pathogenic'`, Athena must:
1. Read **every byte** of every row (including QUAL, FILTER, DP, AF...)
2. Parse the INFO field as a string to extract CLNSIG
3. Then filter

### Parquet is column-oriented
Parquet stores each column separately in the file. To answer the same query:
1. Read **only the CLNSIG column bytes** — skipping everything else
2. Use built-in statistics (min/max per row group) to skip entire blocks

For ClinVar (~2 GB VCF, 30+ columns, ~2M rows), querying just `ClinicalSignificance` might scan:
- **VCF:** 2,000 MB
- **Parquet:** ~50 MB

At $5.00/TB, that's $0.010 vs $0.00025 — a **40x cost difference** for a single query.

## Section 1: Setup

**What this does:** This would install the required Python packages (`boto3`, `awswrangler`, `pandas`, `pyarrow`, `matplotlib`) into the current Python environment. In a real AWS environment these would typically be pre-installed on your SageMaker notebook instance or included in your Glue job's library configuration.

```python
!pip install boto3 awswrangler pandas pyarrow matplotlib --quiet
```

**What this does:** This initializes the AWS session, creates signed and unsigned boto3 S3 clients, and sets the target S3 bucket and Athena results location for the student. It also defines the NCBI FTP URL for downloading the ClinVar variant summary file. Running this cell is a prerequisite for all subsequent cells in the notebook.

```python
import boto3
import awswrangler as wr
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import io
import time
from botocore import UNSIGNED
from botocore.config import Config

STUDENT_ID = "sab032226"   # CHANGE THIS
AWS_REGION = "us-east-1"

BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

s3 = boto3.client('s3', region_name=AWS_REGION)
s3_unsigned = boto3.client('s3', region_name=AWS_REGION,
                            config=Config(signature_version=UNSIGNED))
session = boto3.Session(region_name=AWS_REGION)

# ClinVar variant_summary — direct from NCBI FTP (served over HTTPS, no credentials needed)
# The aws-roda-hcls-datalake public S3 bucket has restricted access; NCBI direct is more reliable.
CLINVAR_NCBI_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"

print("Setup complete.")
```

## Section 2: Exploring the ClinVar Parquet Dataset

The AWS Registry of Open Data hosts ClinVar as pre-converted Parquet — updated weekly. Let's explore its structure.

**What this does:** This would download the first 50,000 rows of the ClinVar `variant_summary.txt.gz` file directly from the NCBI FTP server over HTTPS, parse it as a tab-delimited DataFrame, and normalize all column names to lowercase. The full file is ~500K rows; the 50,000-row subset keeps the download time to about one minute and the in-memory footprint to ~122 MB.

```python
# Download ClinVar variant_summary from NCBI FTP (this may take ~1 minute)
# nrows=50000 keeps the download fast; the full file has ~500K rows

print(f"Downloading ClinVar variant_summary from NCBI...")
print(f"  Source: {CLINVAR_NCBI_URL}\n")

df_clinvar_raw = pd.read_csv(
    CLINVAR_NCBI_URL,
    sep='\t',
    compression='gzip',
    low_memory=False,
    nrows=50000
)

# Normalize column names to lowercase (NCBI uses mixed case like "GeneSymbol", "ClinicalSignificance")
df_clinvar_raw.columns = [
    c.lower().replace(' ', '_').replace('#', '') for c in df_clinvar_raw.columns
]

total_size_mb = df_clinvar_raw.memory_usage(deep=True).sum() / 1024 / 1024
print("ClinVar variant_summary (NCBI):")
print(f"  Rows loaded:    {len(df_clinvar_raw):,}")
print(f"  Columns:        {df_clinvar_raw.shape[1]}")
print(f"  In-memory size: {total_size_mb:.1f} MB")
```

**What this does:** This would print the complete column list and pandas data types for the ClinVar DataFrame. The clean, typed schema (e.g., `clinicalsignificance: object`, `start: int64`) contrasts with a VCF's semi-structured `INFO` field string, illustrating why columnar formats are easier for SQL engines like Athena to work with.

```python
# Inspect the schema — notice how these are clean, typed columns
# compared to VCF's semi-structured INFO field string

print("ClinVar schema (columns):")
for col, dtype in sorted(df_clinvar_raw.dtypes.items()):
    print(f"  {col}: {dtype}")
```

**What this does:** This would display the first five rows of the five most analytically relevant ClinVar columns (`name`, `clinicalsignificance`, `reviewstatus`, `geneid`, `genesymbol`, `chromosome`, `start`) as a pandas DataFrame, giving a concrete preview of the variant data before it is written to S3.

```python
# Preview the key clinical columns
print("Sample rows from ClinVar:")
df_clinvar_raw[['name', 'clinicalsignificance', 'reviewstatus', 'geneid', 'genesymbol', 'chromosome', 'start']].head(5)
```

## Section 3: Creating a Synthetic VCF for Comparison

The ClinVar public data is only in Parquet. To compare fairly against a VCF, we'll write a representative VCF-format version of the same data subset to our S3 bucket, then query both.

**What this does:** This makes a copy of the downloaded ClinVar DataFrame so that the original `df_clinvar_raw` is preserved. The copy is used for all subsequent write operations, ensuring that the data loaded from NCBI is not inadvertently modified. It also prints confirmation of the row count and memory usage.

```python
# Use the data already downloaded in Section 2 — no second download needed
df_clinvar = df_clinvar_raw.copy()

print(f"Loaded {len(df_clinvar):,} rows, {df_clinvar.shape[1]} columns")
print(f"Memory usage: {df_clinvar.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
```

**What this does:** This would encode the ClinVar DataFrame as a tab-delimited string with a minimal VCF-style `##` file header prepended, then write the resulting ~33 MB byte payload to `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=clinvar/clinvar_subset.vcf` using `S3.put_object`. This object represents the row-oriented storage baseline that will be compared against Parquet in the benchmark.

```python
# Write the same data as tab-delimited CSV (simulating VCF row format)
# Real VCF would have ## headers; we add a few for realism

vcf_header = """##fileformat=VCFv4.1
##source=ClinVar
##INFO=<ID=CLNSIG,Number=.,Type=String,Description="Clinical significance">
##INFO=<ID=CLNREVSTAT,Number=.,Type=String,Description="Review status">
##INFO=<ID=GENEINFO,Number=1,Type=String,Description="Gene symbol:GeneID">
"""

# Write VCF-style TSV to S3
vcf_csv_body = vcf_header + df_clinvar.to_csv(sep='\t', index=False)

vcf_key = 'bronze/variants/source=clinvar/clinvar_subset.vcf'
s3.put_object(Bucket=BUCKET, Key=vcf_key, Body=vcf_csv_body.encode())

vcf_size_mb = len(vcf_csv_body.encode()) / 1024 / 1024
print(f"VCF-format file written: {vcf_size_mb:.2f} MB")
print(f"  s3://{BUCKET}/{vcf_key}")
```

**What this does:** This would write the same ClinVar DataFrame as Snappy-compressed Parquet to `s3://binfx410-datalake-{STUDENT_ID}/silver/variants/source=clinvar/` using `awswrangler.s3.to_parquet`, and simultaneously register a Glue Data Catalog table named `binfx410_bronze.clinvar_parquet`. The resulting Parquet file is approximately 5–6 MB — roughly 6x smaller than the equivalent VCF/TSV — because Parquet compresses each column independently and uses dictionary encoding for repeated string values like `clinicalsignificance`.

```python
# Write the same data as Parquet to S3
parquet_path = f"s3://{BUCKET}/silver/variants/source=clinvar/"

wr.s3.to_parquet(
    df=df_clinvar,
    path=parquet_path,
    dataset=True,
    database='binfx410_bronze',
    table='clinvar_parquet',
    boto3_session=session,
    compression='snappy'
)

# Check parquet size
parquet_objects = s3.list_objects_v2(Bucket=BUCKET, Prefix='silver/variants/source=clinvar/')
parquet_size_mb = sum(o['Size'] for o in parquet_objects.get('Contents', [])) / 1024 / 1024

print(f"Parquet file written: {parquet_size_mb:.2f} MB")
print(f"\nSize comparison:")
print(f"  VCF (TSV):  {vcf_size_mb:.2f} MB")
print(f"  Parquet:    {parquet_size_mb:.2f} MB")
print(f"  Ratio:      {vcf_size_mb/parquet_size_mb:.1f}x smaller as Parquet")
```

## Section 4: Registering the VCF as an Athena Table

**What this does:** This would dynamically build a `CREATE EXTERNAL TABLE` DDL statement from the DataFrame's schema and execute it in Athena via `awswrangler.athena.start_query_execution`, registering the VCF/TSV file at `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=clinvar/` as an Athena table named `binfx410_bronze.clinvar_vcf`. The `skip.header.line.count = 6` property tells Athena to skip the six `##` VCF header lines when reading the file.

```python
# Register the VCF-format TSV as an Athena table
# We skip the ## header lines using the TBLPROPERTIES setting

athena = boto3.client('athena', region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)

# Build column definitions from the DataFrame schema
type_map = {'object': 'string', 'int64': 'bigint', 'float64': 'double',
            'bool': 'boolean', 'Int64': 'bigint'}

columns = ', '.join(
    f"`{col}` {type_map.get(str(dtype), 'string')}"
    for col, dtype in df_clinvar.dtypes.items()
    if col not in []  # all columns
)

create_vcf_table = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS binfx410_bronze.clinvar_vcf (
    {columns}
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\\t'
STORED AS TEXTFILE
LOCATION 's3://{BUCKET}/bronze/variants/source=clinvar/'
TBLPROPERTIES (
    'skip.header.line.count' = '6'
)
"""

response = wr.athena.start_query_execution(
    sql=create_vcf_table,
    database='binfx410_bronze',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)
print("VCF table registered in Athena.")
print(f"  Query ID: {response}")
```

## Section 5: Running Benchmark Queries

**What this does:** This defines a `run_benchmark_query` helper function that submits an Athena SQL query, waits for it to complete, retrieves the `DataScannedInBytes` and `EngineExecutionTimeInMillis` statistics from the `GetQueryExecution` API, and returns a dictionary with bytes scanned, wall time, engine time, and computed cost (at $5.00/TB). This function is called twice — once for Parquet and once for VCF — to produce a side-by-side comparison.

```python
# Helper function to run a query and capture performance stats
def run_benchmark_query(sql, database, label):
    """Run an Athena query and return timing + bytes scanned."""
    start = time.time()
    
    query_id = wr.athena.start_query_execution(
        sql=sql,
        database=database,
        s3_output=ATHENA_RESULTS,
        boto3_session=session
    )
    
    # Wait for completion
    df_result = wr.athena.wait_query(query_execution_id=query_id, boto3_session=session)
    elapsed = time.time() - start
    
    # Get query stats
    athena_client = boto3.client('athena', region_name=AWS_REGION)
    stats = athena_client.get_query_execution(QueryExecutionId=query_id)
    bytes_scanned = stats['QueryExecution']['Statistics'].get('DataScannedInBytes', 0)
    engine_ms = stats['QueryExecution']['Statistics'].get('EngineExecutionTimeInMillis', 0)
    
    result = wr.athena.read_sql_query(
        sql=sql, database=database, s3_output=ATHENA_RESULTS, boto3_session=session
    )
    
    return {
        'format': label,
        'bytes_scanned_mb': bytes_scanned / 1024 / 1024,
        'wall_time_sec': elapsed,
        'engine_time_ms': engine_ms,
        'cost_usd': (bytes_scanned / 1e12) * 5.0,
        'result_rows': len(result)
    }

print("Benchmark helper defined.")
```

**What this does:** This would execute the benchmark query (`SELECT clinicalsignificance, COUNT(*) ... WHERE reviewstatus = '...'`) against both `binfx410_bronze.clinvar_parquet` and `binfx410_bronze.clinvar_vcf` in Athena, recording bytes scanned and cost for each. The Parquet query should scan roughly 40x less data than the VCF query because Athena only reads the two needed columns from the column-oriented file, while VCF requires reading every byte of every row. Athena charges $5.00/TB scanned.

```python
# The benchmark query — count variants by clinical significance
# This requires reading: clinicalsignificance, reviewstatus columns
# A good test because: only 2 of ~30 columns are needed

BENCHMARK_SQL_PARQUET = """
SELECT
    clinicalsignificance,
    COUNT(*) AS variant_count
FROM binfx410_bronze.clinvar_parquet
WHERE reviewstatus = 'criteria provided, single submitter'
GROUP BY clinicalsignificance
ORDER BY variant_count DESC
"""

BENCHMARK_SQL_VCF = BENCHMARK_SQL_PARQUET.replace(
    'clinvar_parquet', 'clinvar_vcf'
)

print("Running benchmark on Parquet format...")
stats_parquet = run_benchmark_query(BENCHMARK_SQL_PARQUET, 'binfx410_bronze', 'Parquet')
print(f"  Done: {stats_parquet['bytes_scanned_mb']:.2f} MB scanned")

print("\nRunning benchmark on VCF/TSV format...")
stats_vcf = run_benchmark_query(BENCHMARK_SQL_VCF, 'binfx410_bronze', 'VCF/TSV')
print(f"  Done: {stats_vcf['bytes_scanned_mb']:.2f} MB scanned")

results = pd.DataFrame([stats_parquet, stats_vcf])
print("\n=== Benchmark Results ===")
print(results.to_string(index=False))
```

## Section 6: Visualizing the Difference

**What this does:** This would generate a three-panel matplotlib figure saved as `format_benchmark.png`, comparing VCF vs Parquet across: (1) megabytes scanned, (2) engine execution time in seconds, and (3) query cost in milli-USD. The chart visually communicates the cost and performance advantage of columnar storage to an audience that may not immediately grasp raw numbers.

```python
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('VCF vs Parquet: Query Performance Comparison\n(ClinVar — 50,000 variant subset)',
             fontsize=13, fontweight='bold')

colors = ['#2196F3', '#4CAF50']  # Blue for VCF, Green for Parquet
formats = results['format'].tolist()

# Plot 1: Bytes scanned
ax1 = axes[0]
bars = ax1.bar(formats, results['bytes_scanned_mb'], color=colors)
ax1.set_title('Data Scanned (MB)', fontweight='bold')
ax1.set_ylabel('Megabytes')
for bar, val in zip(bars, results['bytes_scanned_mb']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f} MB', ha='center', fontweight='bold')

# Plot 2: Query time
ax2 = axes[1]
bars2 = ax2.bar(formats, results['engine_time_ms'] / 1000, color=colors)
ax2.set_title('Engine Execution Time (sec)', fontweight='bold')
ax2.set_ylabel('Seconds')
for bar, val in zip(bars2, results['engine_time_ms'] / 1000):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}s', ha='center', fontweight='bold')

# Plot 3: Cost
ax3 = axes[2]
bars3 = ax3.bar(formats, results['cost_usd'] * 1000, color=colors)  # millicents
ax3.set_title('Query Cost (milli-USD)', fontweight='bold')
ax3.set_ylabel('USD × 10⁻³')
for bar, val in zip(bars3, results['cost_usd'] * 1000):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'${val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('format_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

ratio = results.loc[results['format']=='VCF/TSV', 'bytes_scanned_mb'].values[0] / \
        results.loc[results['format']=='Parquet', 'bytes_scanned_mb'].values[0]
print(f"\nParquet scanned {ratio:.1f}x LESS data than VCF for the same query.")
```

## Section 7: The Cost Calculator

**What this does:** This would extrapolate the benchmark results to a realistic production workload: scaling the per-query scan bytes to the full 2 GB ClinVar VCF, then multiplying by 50 queries/day and 30 days/month. The output shows a concrete monthly cost comparison — expected to be approximately $14.65/month for VCF vs $0.02/month for Parquet — making the business case for format conversion tangible.

```python
# Extrapolate costs to a realistic full ClinVar dataset and a genomics lab workload

ATHENA_PRICE_PER_TB = 5.00

# Assume a genomics lab runs 50 similar queries per day
queries_per_day = 50
days_per_month = 30

# Full ClinVar is ~2 GB VCF — scale our benchmark proportionally
scale_factor = 2048 / vcf_size_mb  # ratio of full ClinVar to our subset

for row in [stats_vcf, stats_parquet]:
    fmt = row['format']
    single_query_mb = row['bytes_scanned_mb'] * scale_factor
    single_query_cost = (single_query_mb / 1024 / 1024) * ATHENA_PRICE_PER_TB
    monthly_cost = single_query_cost * queries_per_day * days_per_month
    
    print(f"Format: {fmt}")
    print(f"  Bytes scanned per query (full ClinVar): {single_query_mb:.0f} MB")
    print(f"  Cost per query: ${single_query_cost:.4f}")
    print(f"  Monthly cost ({queries_per_day} queries/day): ${monthly_cost:.2f}")
    print()
```

## Section 8: Converting VCF to Parquet

**What this does:** This would download the VCF/TSV file from `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=clinvar/clinvar_subset.vcf` using `S3.get_object`, filter out the `##` meta-header lines, and parse the remaining content into a pandas DataFrame. This simulates the Bronze→Silver extraction step where a raw row-oriented file is read in preparation for conversion to a columnar format.

```python
# Read back our VCF from S3 and convert to Parquet
# This is the Bronze → Silver transformation for variant data

obj = s3.get_object(Bucket=BUCKET, Key='bronze/variants/source=clinvar/clinvar_subset.vcf')
raw_vcf = obj['Body'].read().decode()

# Skip ## header lines
data_lines = [l for l in raw_vcf.split('\n') if l and not l.startswith('##')]
df_from_vcf = pd.read_csv(io.StringIO('\n'.join(data_lines)), sep='\t', low_memory=False)

print(f"Parsed {len(df_from_vcf):,} rows from VCF")
print(f"Columns: {list(df_from_vcf.columns[:8])} ...")
```

**What this does:** This would serialize the DataFrame as a Snappy-compressed Parquet file using `pyarrow` via `df.to_parquet`, then upload the bytes to `s3://binfx410-datalake-{STUDENT_ID}/silver/variants/source=clinvar/clinvar_converted.parquet` using `S3.put_object`. It then compares the S3 object sizes of the source VCF and the converted Parquet, printing the compression ratio. This represents the Bronze→Silver promotion step in the medallion architecture.

```python
# Write to Parquet with explicit schema — best practice to avoid type inference issues
# In real pipelines, you'd define this schema in a config file

parquet_buffer = io.BytesIO()
df_from_vcf.to_parquet(parquet_buffer, engine='pyarrow', compression='snappy', index=False)
parquet_buffer.seek(0)

s3.put_object(
    Bucket=BUCKET,
    Key='silver/variants/source=clinvar/clinvar_converted.parquet',
    Body=parquet_buffer.read()
)

# Compare sizes
vcf_obj = s3.head_object(Bucket=BUCKET, Key='bronze/variants/source=clinvar/clinvar_subset.vcf')
parquet_obj = s3.head_object(Bucket=BUCKET, Key='silver/variants/source=clinvar/clinvar_converted.parquet')

vcf_mb = vcf_obj['ContentLength'] / 1024 / 1024
parquet_mb = parquet_obj['ContentLength'] / 1024 / 1024

print("Conversion result:")
print(f"  VCF size:     {vcf_mb:.2f} MB")
print(f"  Parquet size: {parquet_mb:.2f} MB")
print(f"  Compression:  {(1 - parquet_mb/vcf_mb)*100:.1f}% smaller")
```

## Exercise: YOUR CODE HERE

**What this does:** This is a student exercise scaffold. When completed, it would run a second benchmark Athena query that accesses more columns (`genesymbol`, `clinicalsignificance`, `chromosome`, `start`) against both `clinvar_parquet` and `clinvar_vcf` tables to find the top 10 genes by pathogenic variant count. Because more columns are needed, the relative advantage of Parquet over VCF is expected to narrow slightly, demonstrating that columnar storage benefits are greatest when queries touch a small subset of columns.

```python
# TASK 1: Run a second benchmark query that accesses MORE columns
# Query: find the top 10 genes by number of pathogenic variants,
# including columns: genesymbol, clinicalsignificance, chromosome, start
# Do this for both Parquet and VCF/TSV tables
# Predict: will the gap be larger or smaller than the first query? Why?

# YOUR CODE HERE
multi_col_sql_parquet = """
-- YOUR QUERY HERE (against clinvar_parquet)
"""

multi_col_sql_vcf = """
-- YOUR QUERY HERE (against clinvar_vcf)
"""
```

**What this does:** This is a student exercise scaffold. When completed, it would combine the results of all four benchmark runs (two queries × two formats) into a single pandas summary DataFrame with columns for format, query name, bytes scanned, and cost. This structured summary makes it easy to create comparative visualizations and to reason about Athena cost at scale.

```python
# TASK 2: Write the benchmark results to a summary DataFrame
# Include: format, query_name, bytes_scanned_mb, cost_usd
# for all 4 queries (2 formats × 2 queries)

# YOUR CODE HERE
summary_df = pd.DataFrame()
print(summary_df)
```

**What this does:** This is a student exercise scaffold. When completed, it would use matplotlib to render a grouped bar chart with query name on the x-axis and format (VCF vs Parquet) as bar color, showing bytes scanned side by side for both benchmark queries. This visualization reinforces how the columnar advantage scales with the number of columns accessed.

```python
# TASK 3: Create a bar chart comparing bytes scanned for both queries, both formats
# Use grouped bars (query on x-axis, format as color)

# YOUR CODE HERE
```

## Reflection Questions

*(Double-click to edit)*

1. **The INFO field in VCF is a semi-structured string** (`DP=42;AF=0.45;CLNSIG=Pathogenic`). What specific challenge does this create for Athena? How does Parquet solve it?

2. **Columnar storage helps most when you query few columns.** If a genomics analysis needs ALL columns of every variant (e.g., a full-genome GWAS export), would Parquet still be faster than VCF? Why or why not?

3. **VCF is not going away.** Name three bioinformatics tools or workflows that still require VCF format. What does this suggest about where in the medallion architecture VCF belongs?

4. **Row group statistics:** Parquet files store min/max values per column per row group. How could this help a query `WHERE start BETWEEN 1000000 AND 2000000` on a genomic coordinate? What would need to be true about how the file was written for this optimization to work?

5. **Your lab has 50 TB of VCF files** accumulated over 10 years. A researcher wants to run monthly population-level queries over all of them. Make the business case (in dollars) for converting them to Parquet, assuming 100 queries/month at average 500 GB scanned per query.

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 